In [1]:
# Requirements: python>=3.8, numpy, torch. Experiment 7 only: transformers and internet access (model download).
import os, sys, json, time, math, random, hashlib, platform
import numpy as np
import torch

SEEDS = [42, 123, 456]
OUT_DIR = "/kaggle/working/dqn_society_run2" if os.path.isdir("/kaggle/working") else "./dqn_society_run2"
RESULTS_FILE = "run2_results.jsonl"
CODE_VERSION = "r2v1"
TIME_BUDGET_HOURS = 11.0
LLM_MODELS = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-0.5B-Instruct"]
LLM_BATCH = 64
LLM_TEMPERATURE = 1.0
LLM_HOURS_MAX = 5.0
ACTIONS = ("collect", "invest", "reproduce")
T_START = time.time()

BASE = dict(
    env="paper", system="free", groups=0, split="edu6040", policy="dqn",
    n_agents=50, days=365, iterations=3, food_lo=1000, food_hi=3000,
    max_age_lo=365, max_age_hi=3650, wealth_lo=10, wealth_hi=50, edu_lo=0.0, edu_hi=30.0,
    needs_lo=10, needs_hi=40, days_until_death=2, invest_base=0.5, invest_gain=0.1,
    repro_wealth=200, cooldown=365, collect_eff=0.08, edu_gain_collect=0.01, edu_gain_pool=0.005,
    partner_prob=0.15, lr=1e-3, gamma=0.99, eps_start=0.9, eps_end=0.05, eps_decay_days=730,
    eps_clock="iteration", cooldown_clock="iteration", buffer=1000, batch=32, hidden=64,
    target_update=100, grad_clip=100.0, r_invest_success=0.2, r_invest_fail=-0.1, r_repro=0.5,
    happiness_term=True, clip_state=False, pop_cap=3000,
    ext_invest_win=0.5, ext_invest_loss=0.5, ext_wealth_scale=100.0, ext_hunger_penalty=0.5,
    ext_death_penalty=1.0, inherit_net=False, donor=None, save_bank=False, llm_interval=1)

EXT = dict(env="ext", clip_state=True, eps_clock="global", cooldown_clock="global")
SYSTEMS = [("free", dict(system="free")), ("centralized", dict(system="centralized", groups=1, split="edu6040")),
           ("hybrid5", dict(system="hybrid", groups=5, split="equal")),
           ("hybrid10", dict(system="hybrid", groups=10, split="equal"))]
LLM = {"obj": None, "name": None}


def cfg_of(*dicts, **kw):
    c = dict(BASE)
    for d in dicts:
        c.update(d)
    c.update(kw)
    return c


def run_key(cfg, seed):
    blob = json.dumps(dict(cfg=cfg, seed=seed, v=CODE_VERSION, llm=LLM["name"] if cfg["policy"] == "llm" else None),
                      sort_keys=True)
    return hashlib.md5(blob.encode()).hexdigest()[:16]


def gini(values):
    v = np.sort(np.asarray(values, dtype=np.float64))
    n = len(v)
    if n == 0 or v.sum() <= 0:
        return 0.0
    idx = np.arange(1, n + 1)
    return float((2 * np.sum(idx * v)) / (n * v.sum()) - (n + 1) / n)


class Bank:
    def __init__(self, c, cap=128):
        h = c["hidden"]
        self.c, self.shapes = c, [(5, h), (1, h), (h, h), (1, h), (h, 3), (1, 3)]
        self.P = [torch.zeros(0, *s) for s in self.shapes]
        self.T = [torch.zeros(0, *s) for s in self.shapes]
        self.M = [torch.zeros(0, *s) for s in self.shapes]
        self.V = [torch.zeros(0, *s) for s in self.shapes]
        self.t = np.zeros(0, dtype=np.int64)
        self.cnt = np.zeros(0, dtype=np.int64)
        self.pos = np.zeros(0, dtype=np.int64)
        self.mem = np.zeros((0, c["buffer"], 13), dtype=np.float32)
        self.cap, self.free = 0, []
        self.grow(cap)

    def grow(self, n):
        for L in (self.P, self.T, self.M, self.V):
            for i, s in enumerate(self.shapes):
                L[i] = torch.cat([L[i], torch.zeros(n, *s)], 0)
        self.t = np.concatenate([self.t, np.zeros(n, dtype=np.int64)])
        self.cnt = np.concatenate([self.cnt, np.zeros(n, dtype=np.int64)])
        self.pos = np.concatenate([self.pos, np.zeros(n, dtype=np.int64)])
        self.mem = np.concatenate([self.mem, np.zeros((n, self.c["buffer"], 13), dtype=np.float32)], 0)
        self.free.extend(range(self.cap + n - 1, self.cap - 1, -1))
        self.cap += n

    def alloc(self, src=None):
        if not self.free:
            self.grow(self.cap)
        k = self.free.pop()
        for i, s in enumerate(self.shapes):
            if src is None:
                bound = 1.0 / math.sqrt(self.shapes[i - (i % 2)][0])
                self.P[i][k].uniform_(-bound, bound)
            else:
                self.P[i][k].copy_(src[i])
            self.T[i][k].copy_(self.P[i][k])
            self.M[i][k].zero_()
            self.V[i][k].zero_()
        self.t[k] = self.cnt[k] = self.pos[k] = 0
        return k

    def params_of(self, k):
        return [p[k].clone() for p in self.P]

    @staticmethod
    def fwd(P, x):
        h = torch.relu(torch.bmm(x, P[0]) + P[1])
        h = torch.relu(torch.bmm(h, P[2]) + P[3])
        return torch.bmm(h, P[4]) + P[5]

    def q(self, slots, X):
        with torch.no_grad():
            idx = torch.as_tensor(slots, dtype=torch.long)
            x = torch.as_tensor(np.asarray(X, dtype=np.float32)).unsqueeze(1)
            return self.fwd([p[idx] for p in self.P], x).squeeze(1).numpy()

    def push(self, k, s, a, r, s2, done):
        row = self.mem[k, self.pos[k]]
        row[0:5], row[5], row[6], row[7:12], row[12] = s, a, r, s2, 1.0 if done else 0.0
        self.pos[k] = (self.pos[k] + 1) % self.c["buffer"]
        self.cnt[k] = min(self.cnt[k] + 1, self.c["buffer"])

    def optimize(self, slots):
        c = self.c
        B, buf = c["batch"], c["buffer"]
        sl = np.array([k for k in slots if self.cnt[k] >= B], dtype=np.int64)
        n = len(sl)
        if n == 0:
            return []
        r = np.random.rand(n, buf)
        r[np.arange(buf)[None, :] >= self.cnt[sl][:, None]] = 2.0
        bi = np.argpartition(r, B - 1, axis=1)[:, :B]
        batch = torch.from_numpy(self.mem[sl[:, None], bi])
        S, A, R = batch[:, :, 0:5], batch[:, :, 5:6].long(), batch[:, :, 6:7]
        S2, ND = batch[:, :, 7:12], 1.0 - batch[:, :, 12:13]
        idx = torch.from_numpy(sl)
        P = [p[idx].clone().requires_grad_(True) for p in self.P]
        qv = self.fwd(P, S).gather(2, A)
        with torch.no_grad():
            tgt = R + c["gamma"] * self.fwd([p[idx] for p in self.T], S2).max(2, keepdim=True)[0] * ND
        li = ((qv - tgt) ** 2).mean(dim=(1, 2))
        g = torch.autograd.grad(li.sum(), P)
        self.t[sl] += 1
        tt = torch.from_numpy(self.t[sl]).double()
        c1 = (1 - 0.9 ** tt).float().view(-1, 1, 1)
        c2 = (1 - 0.999 ** tt).float().view(-1, 1, 1)
        for i in range(6):
            gi = g[i].clamp(-c["grad_clip"], c["grad_clip"])
            m = self.M[i][idx] * 0.9 + 0.1 * gi
            v = self.V[i][idx] * 0.999 + 0.001 * gi * gi
            self.M[i][idx], self.V[i][idx] = m, v
            self.P[i][idx] = P[i].detach() - c["lr"] * (m / c1) / ((v / c2).sqrt() + 1e-8)
        upd = sl[self.t[sl] % c["target_update"] == 0]
        if len(upd):
            ui = torch.from_numpy(upd)
            for i in range(6):
                self.T[i][ui] = self.P[i][ui]
        return li.detach().numpy().tolist()


class Agent:
    __slots__ = ("id", "group_id", "age", "max_age", "wealth", "education", "needs", "children", "happiness", "dwf",
                 "alive", "last_repro", "cohort0", "slot", "c", "short", "w0", "last_act", "last_perm")

    def __init__(self, aid, c, group_id=0, cohort0=False):
        self.id, self.group_id, self.c, self.cohort0 = aid, group_id, c, cohort0
        self.age = random.randint(0, 100)
        self.max_age = random.randint(c["max_age_lo"], c["max_age_hi"])
        self.wealth = random.randint(c["wealth_lo"], c["wealth_hi"])
        self.education = random.uniform(c["edu_lo"], c["edu_hi"])
        self.needs = random.randint(c["needs_lo"], c["needs_hi"])
        self.children, self.happiness, self.dwf, self.alive = 0, 0.0, 0, True
        self.last_repro = -c["cooldown"]
        self.slot, self.short, self.w0, self.last_act, self.last_perm = -1, 0.0, 0.0, None, None
        self.calc_happiness()

    def calc_happiness(self):
        self.happiness = (min(self.wealth / 1000.0, 1.0) * 0.4 + (self.education / 100.0) * 0.4
                          + min(self.children / 5.0, 1.0) * 0.2)
        return self.happiness

    def state(self, food):
        s = [self.wealth / 10000.0, self.age / float(self.c["max_age_hi"]), self.education / 100.0,
             self.needs / 100.0, food / 3000.0]
        if self.c["clip_state"]:
            s = [min(max(x, 0.0), 1.0) for x in s]
        return s

    def can_reproduce(self, clock_day):
        return self.wealth >= self.c["repro_wealth"] and clock_day - self.last_repro >= self.c["cooldown"]


LLM_SYSTEM = ("You are a person living in a small society. Each day you choose exactly one activity. "
              "Reply with a single letter only.")
OPTION_TEXT = (
    "Collect food: spend the day gathering food.",
    "Invest: put up to a tenth of your wealth into a venture that can gain or lose; more education improves the odds.",
    "Start a family: try to have a child (requires at least {rw} wealth and a willing partner, at most once a year).")


class LLMPolicy:
    def __init__(self, name):
        from transformers import AutoTokenizer, AutoModelForCausalLM
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.tok = AutoTokenizer.from_pretrained(name)
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        dtype = torch.float16 if self.dev == "cuda" else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(self.dev).eval()
        self.ids = [self.tok.encode(x, add_special_tokens=False)[0] for x in "ABC"]
        self.n_prompts, self.seconds = 0, 0.0

    def wrap(self, user):
        if getattr(self.tok, "chat_template", None):
            return self.tok.apply_chat_template([{"role": "system", "content": LLM_SYSTEM},
                                                 {"role": "user", "content": user}],
                                                tokenize=False, add_generation_prompt=True)
        return LLM_SYSTEM + "\n" + user + "\nAnswer:"

    def probs(self, users):
        t0 = time.time()
        texts, res = [self.wrap(u) for u in users], []
        for i in range(0, len(texts), LLM_BATCH):
            enc = self.tok(texts[i:i + LLM_BATCH], return_tensors="pt", padding=True).to(self.dev)
            with torch.no_grad():
                h = self.model.base_model(**enc).last_hidden_state[:, -1, :]
                lg = self.model.get_output_embeddings()(h)[:, self.ids]
            res.append(torch.softmax(lg.float() / LLM_TEMPERATURE, dim=-1).cpu())
        self.n_prompts += len(texts)
        self.seconds += time.time() - t0
        return torch.cat(res).numpy()


class StubLLM:
    n_prompts, seconds = 0, 0.0

    def probs(self, users):
        self.n_prompts += len(users)
        return np.full((len(users), 3), 1.0 / 3.0)


class Society:
    def __init__(self, c, donors=None):
        self.c = c
        self.pooled = c["system"] in ("centralized", "hybrid")
        self.n_groups = 1 if c["system"] == "centralized" else (c["groups"] if c["system"] == "hybrid" else 0)
        self.learn = c["policy"] == "dqn"
        self.neural = c["policy"] in ("dqn", "frozen")
        self.bank = Bank(c) if self.neural else None
        self.donors = donors
        self.agents = [Agent(i, c, cohort0=True) for i in range(c["n_agents"])]
        self.groups = [[] for _ in range(self.n_groups)]
        for i, ag in enumerate(self.agents):
            if self.pooled:
                ag.group_id = i % self.n_groups
                self.groups[ag.group_id].append(ag)
            if self.neural:
                ag.slot = self.bank.alloc(self.donor_params())
        self.next_id = c["n_agents"]
        self.total_births = 0
        self.total_deaths = dict(starvation=0, age=0)
        self.cap_hit = False
        self.reset_flow()

    def donor_params(self):
        if self.donors is None:
            return None
        j = random.randrange(self.donors[0].shape[0])
        return [d[j] for d in self.donors]

    def reset_flow(self):
        self.flow = dict(births=0, d_starv=0, d_age=0, hungry=0, loss_sum=0.0, loss_n=0, q_sum=0.0, q_n=0, r_sum=0.0,
                         r_n=0, act=[0, 0, 0], greedy=[0, 0, 0])

    def epsilon(self, day, gday):
        c = self.c
        if c["policy"] == "frozen":
            return 0.0
        d = day if c["eps_clock"] == "iteration" else gday
        if d < c["eps_decay_days"]:
            return c["eps_start"] - (c["eps_start"] - c["eps_end"]) * (d / c["eps_decay_days"])
        return c["eps_end"]

    def prompt(self, ag, food, perm):
        c = self.c
        if not self.pooled:
            soc = "You live on your own. Nobody shares food with you."
        else:
            n = sum(1 for a in self.groups[ag.group_id] if a.alive)
            rule = ("most of any surplus goes to the best educated members" if c["split"] == "edu6040"
                    else "any surplus is shared equally")
            if c["env"] == "ext":
                soc = (f"You belong to a group of {n} people with a common food store. Only members who collect food "
                       f"that day add to the store; the store feeds every member and {rule}.")
            else:
                soc = f"You belong to a group of {n} people that gathers food together and shares it; {rule}."
        opts = "\n".join(f"{'ABC'[i]}) {OPTION_TEXT[a].format(rw=c['repro_wealth'])}" for i, a in enumerate(perm))
        return (f"Your situation today:\n- Age: {ag.age} days (people here live at most about {c['max_age_hi']} days)\n"
                f"- Wealth: {ag.wealth:.0f} units\n- Education: {ag.education:.0f} out of 100\n"
                f"- Food you need every day: {ag.needs} units\n- Food available in the environment today: {food} units\n"
                f"- Children: {ag.children}\n- Days in a row without enough food: {ag.dwf}\n{soc}\n"
                f"Options:\n{opts}\nWhich option do you choose? Answer with A, B or C.")

    def decide(self, live, S, day, gday, clock_day, food):
        pol, f = self.c["policy"], self.flow
        if self.neural:
            q = self.bank.q([a.slot for a in live], S)
            g = q.argmax(1)
            f["q_sum"] += float(q.max(1).sum())
            f["q_n"] += len(live)
            eps = self.epsilon(day, gday)
            acts = []
            for i in range(len(live)):
                f["greedy"][int(g[i])] += 1
                acts.append(int(g[i]) if random.random() > eps else random.randint(0, 2))
            return acts
        if pol == "random":
            return [random.randint(0, 2) for _ in live]
        if pol == "rule":
            return [2 if a.can_reproduce(clock_day) else (0 if a.wealth < self.c["repro_wealth"] else 1) for a in live]
        if pol == "forager":
            return [2 if a.can_reproduce(clock_day) else 0 for a in live]
        ask = [a for a in live if a.last_act is None or gday % self.c["llm_interval"] == 0]
        if ask:
            perms = [random.sample([0, 1, 2], 3) for _ in ask]
            pr = LLM["obj"].probs([self.prompt(a, food, p) for a, p in zip(ask, perms)])
            for a, p, row in zip(ask, perms, pr):
                row = np.asarray(row, dtype=np.float64)
                row = row / row.sum()
                a.last_act = p[int(np.searchsorted(np.cumsum(row), random.random() * 0.999999))]
                f["greedy"][p[int(row.argmax())]] += 1
        return [a.last_act for a in live]

    def share_surplus(self, alive, surplus):
        if self.c["split"] == "edu6040":
            ranked = sorted(alive, key=lambda a: a.education, reverse=True)
            top = max(1, len(ranked) // 5)
            for i, ag in enumerate(ranked):
                if i < top:
                    ag.wealth += surplus * 0.6 / top
                elif len(ranked) > top:
                    ag.wealth += surplus * 0.4 / (len(ranked) - top)
        else:
            for ag in alive:
                ag.wealth += surplus / len(alive)

    def pool_paper(self, food):
        c, f = self.c, self.flow
        for group in self.groups:
            alive = [a for a in group if a.alive]
            if not alive:
                continue
            coll = []
            for ag in alive:
                coll.append((ag, random.uniform(0, food * c["collect_eff"])))
                ag.education += c["edu_gain_pool"]
            total = sum(x for _, x in coll)
            needs = sum(a.needs for a in alive)
            if total >= needs:
                self.share_surplus(alive, total - needs)
                for ag in alive:
                    ag.dwf = 0
            else:
                ratio = total / needs if needs > 0 else 0
                for ag, fc in coll:
                    if fc >= ag.needs * ratio:
                        ag.dwf = 0
                    else:
                        ag.dwf += 1
                        if ag.dwf >= c["days_until_death"]:
                            ag.alive = False
                            f["d_starv"] += 1

    def pool_ext(self, live, acts, food):
        c = self.c
        act_of = {id(a): x for a, x in zip(live, acts)}
        for group in self.groups:
            alive = [a for a in group if a.alive]
            if not alive:
                continue
            total = 0.0
            for ag in alive:
                if act_of.get(id(ag)) == 0:
                    total += random.uniform(0, food * c["collect_eff"])
                    ag.education += c["edu_gain_pool"]
            needs = sum(a.needs for a in alive)
            if total >= needs:
                self.share_surplus(alive, total - needs)
                for ag in alive:
                    ag.short = 0.0
            else:
                ratio = total / needs if needs > 0 else 0.0
                for ag in alive:
                    ag.short = ag.needs * (1.0 - ratio)

    def step_day(self, day, gday):
        c, f = self.c, self.flow
        ext = c["env"] == "ext"
        food = random.randint(c["food_lo"], c["food_hi"])
        clock_day = day if c["cooldown_clock"] == "iteration" else gday
        if self.pooled and not ext:
            self.pool_paper(food)
        live = [a for a in self.agents if a.alive]
        newborn = []
        if live:
            S = [a.state(food) for a in live]
            acts = self.decide(live, S, day, gday, clock_day, food)
            for ag in live:
                ag.w0 = ag.wealth
            if self.pooled and ext:
                self.pool_ext(live, acts, food)
            for ag, s, act in zip(live, S, acts):
                f["act"][act] += 1
                reward, hungry, starved, born = 0.0, False, False, False
                if ext:
                    if self.pooled:
                        short = ag.short
                    else:
                        got = 0.0
                        if act == 0:
                            got = random.uniform(0, food * c["collect_eff"])
                            ag.education += c["edu_gain_collect"]
                        short = ag.needs - got
                        if short <= 0:
                            ag.wealth -= short
                    if short > 0 and ag.wealth >= short:
                        ag.wealth -= short
                        short = 0.0
                    if short > 0:
                        ag.dwf += 1
                        hungry = True
                        f["hungry"] += 1
                        starved = ag.dwf >= c["days_until_death"]
                    else:
                        ag.dwf = 0
                elif act == 0 and not self.pooled:
                    got = random.uniform(0, food * c["collect_eff"])
                    if got >= ag.needs:
                        ag.wealth += got - ag.needs
                        ag.education += c["edu_gain_collect"]
                        ag.dwf = 0
                    else:
                        ag.dwf += 1
                        f["hungry"] += 1
                        if ag.dwf >= c["days_until_death"]:
                            ag.alive = False
                            f["d_starv"] += 1
                            continue
                if act == 1 and not starved:
                    amount = min(ag.wealth * 0.1, 100)
                    p = min(1.0, c["invest_base"] + ag.education / 100.0)
                    win = random.random() < p
                    if ext:
                        ag.wealth += amount * c["ext_invest_win"] if win else -amount * c["ext_invest_loss"]
                    elif win:
                        ag.wealth += amount * c["invest_gain"]
                        reward = c["r_invest_success"]
                    else:
                        ag.wealth -= amount
                        reward = c["r_invest_fail"]
                elif act == 2 and not starved and ag.can_reproduce(clock_day):
                    pool = self.groups[ag.group_id] if self.pooled else self.agents
                    partners = [a for a in pool if a is not ag and a.alive and a.can_reproduce(clock_day)
                                and random.random() < c["partner_prob"]]
                    if partners and (len(self.agents) + len(newborn)) < c["pop_cap"]:
                        partner = random.choice(partners)
                        child = Agent(self.next_id, c, group_id=ag.group_id)
                        self.next_id += 1
                        child.education = max(0, min((ag.education + partner.education) / 2 + random.uniform(-5, 5), 100))
                        child.wealth = max(0, (ag.wealth + partner.wealth) / 4)
                        ag.children += 1
                        partner.children += 1
                        ag.last_repro = partner.last_repro = clock_day
                        if self.neural:
                            inherit = c["inherit_net"] or c["policy"] == "frozen"
                            child.slot = self.bank.alloc(self.bank.params_of(ag.slot) if inherit else None)
                        newborn.append(child)
                        f["births"] += 1
                        born = True
                        if not ext:
                            reward = c["r_repro"]
                    elif partners:
                        self.cap_hit = True
                old_h = ag.happiness
                ag.age += 1
                if starved:
                    ag.alive, done = False, True
                    f["d_starv"] += 1
                elif ag.age >= ag.max_age:
                    ag.alive, done = False, True
                    f["d_age"] += 1
                else:
                    ag.calc_happiness()
                    done = False
                s2 = [0.0] * 5 if done else ag.state(food)
                if ext:
                    reward = max(-1.0, min(1.0, (ag.wealth - ag.w0) / c["ext_wealth_scale"]))
                    reward += (c["r_repro"] if born else 0.0) - (c["ext_hunger_penalty"] if hungry else 0.0)
                    reward -= c["ext_death_penalty"] if starved else 0.0
                if c["happiness_term"]:
                    reward += ag.happiness - old_h
                f["r_sum"] += reward
                f["r_n"] += 1
                if self.learn:
                    self.bank.push(ag.slot, s, act, reward, s2, done)
            if self.learn:
                losses = self.bank.optimize([a.slot for a in live if a.alive])
                f["loss_sum"] += float(sum(losses))
                f["loss_n"] += len(losses)
        if self.neural:
            for a in self.agents:
                if not a.alive and a.slot >= 0:
                    self.bank.free.append(a.slot)
                    a.slot = -1
        self.agents = [a for a in self.agents if a.alive]
        for ch in newborn:
            self.agents.append(ch)
            if self.pooled:
                self.groups[ch.group_id].append(ch)
        if self.pooled:
            self.groups = [[a for a in g if a.alive] for g in self.groups]

    def summary(self, fl):
        c = self.c
        w = np.array([a.wealth for a in self.agents], dtype=np.float64)
        h = np.array([a.happiness for a in self.agents], dtype=np.float64)
        e = np.array([a.education for a in self.agents], dtype=np.float64)
        n = len(w)
        ever = c["n_agents"] + self.total_births
        dead = self.total_deaths["starvation"] + self.total_deaths["age"]
        cohort_alive = sum(1 for a in self.agents if a.cohort0)
        r = dict(pop=n, births_iter=fl["births"], deaths_starv_iter=fl["d_starv"], deaths_age_iter=fl["d_age"],
                 hungry_days_iter=fl["hungry"], births_cum=self.total_births, deaths_cum=dead,
                 deaths_starv_cum=self.total_deaths["starvation"], deaths_age_cum=self.total_deaths["age"],
                 ever_lived=ever, mortality_ever_lived=dead / ever if ever else 0.0, cohort0_survivors=cohort_alive,
                 cohort0_mortality=1 - cohort_alive / c["n_agents"])
        names = ("wealth_mean", "wealth_median", "wealth_std", "wealth_min", "wealth_max", "gini", "gini_ever_lived",
                 "top20_share", "happy_mean", "happy_std", "happy_mean_ever_lived", "edu_mean", "edu_std")
        if n:
            ws = np.sort(w)
            top = max(1, n // 5)
            vals = (w.mean(), np.median(w), w.std(), w.min(), w.max(), gini(w),
                    gini(np.concatenate([w, np.zeros(ever - n)])), ws[-top:].sum() / ws.sum() if ws.sum() > 0 else 0.0,
                    h.mean(), h.std(), h.sum() / ever, e.mean(), e.std())
        else:
            vals = (0.0,) * len(names)
        r.update({k: float(v) for k, v in zip(names, vals)})
        ta, tg = sum(fl["act"]), sum(fl["greedy"])
        for i, nm in enumerate(ACTIONS):
            r[f"act_{nm}"] = fl["act"][i] / ta if ta else 0.0
            r[f"greedy_{nm}"] = fl["greedy"][i] / tg if tg else None
        r["loss_mean"] = fl["loss_sum"] / fl["loss_n"] if fl["loss_n"] else None
        r["maxq_mean"] = fl["q_sum"] / fl["q_n"] if fl["q_n"] else None
        r["reward_mean"] = fl["r_sum"] / fl["r_n"] if fl["r_n"] else None
        r["agent_steps"] = fl["r_n"]
        return r, w


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def simulate(cfg, seed, donors=None):
    seed_all(seed)
    t0 = time.time()
    soc = Society(cfg, donors)
    summaries, series, gday = [], [], 0
    keys = ("births", "d_starv", "d_age", "hungry", "loss_sum", "loss_n", "q_sum", "q_n", "r_sum", "r_n")
    w = np.zeros(0)
    for it in range(1, cfg["iterations"] + 1):
        itf = dict({k: 0 for k in keys}, act=[0, 0, 0], greedy=[0, 0, 0])
        soc.reset_flow()
        for day in range(cfg["days"]):
            soc.step_day(day, gday)
            gday += 1
            if day % 30 == 29 or day == cfg["days"] - 1:
                f = soc.flow
                ws = [a.wealth for a in soc.agents]
                n = len(ws)
                ta, tg = sum(f["act"]), sum(f["greedy"])
                series.append(dict(
                    it=it, day=day, pop=n, births=f["births"], d_starv=f["d_starv"], d_age=f["d_age"], hungry=f["hungry"],
                    wealth=float(np.mean(ws)) if n else 0.0, gini=gini(ws) if n else 0.0,
                    happy=float(np.mean([a.happiness for a in soc.agents])) if n else 0.0,
                    edu=float(np.mean([a.education for a in soc.agents])) if n else 0.0,
                    eps=soc.epsilon(day, gday - 1), loss=f["loss_sum"] / f["loss_n"] if f["loss_n"] else None,
                    maxq=f["q_sum"] / f["q_n"] if f["q_n"] else None, reward=f["r_sum"] / f["r_n"] if f["r_n"] else None,
                    act=[x / ta if ta else 0.0 for x in f["act"]], greedy=[x / tg if tg else None for x in f["greedy"]]))
                for k in keys:
                    itf[k] += f[k]
                for i in range(3):
                    itf["act"][i] += f["act"][i]
                    itf["greedy"][i] += f["greedy"][i]
                soc.total_births += f["births"]
                soc.total_deaths["starvation"] += f["d_starv"]
                soc.total_deaths["age"] += f["d_age"]
                soc.reset_flow()
        summ, w = soc.summary(itf)
        summ["it"] = it
        summaries.append(summ)
    bank = None
    if cfg["save_bank"] and soc.neural and soc.agents:
        idx = torch.as_tensor([a.slot for a in soc.agents], dtype=torch.long)
        bank = [p[idx].clone() for p in soc.bank.P]
    res = dict(summaries=summaries, series=series, final_wealth=[round(float(x), 1) for x in np.sort(w)],
               wall=time.time() - t0, steps=sum(s["agent_steps"] for s in summaries), cap_hit=soc.cap_hit)
    return res, bank


DONE = {}


def load_done():
    path = os.path.join(OUT_DIR, RESULTS_FILE)
    if os.path.exists(path):
        with open(path) as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                    DONE[rec["key"]] = rec
                except Exception:
                    pass
    print(f"RESUME records_found={len(DONE)} file={path}", flush=True)


def bank_path(key):
    return os.path.join(OUT_DIR, f"bank_{key}.pt")


def over_budget():
    return (time.time() - T_START) / 3600.0 > TIME_BUDGET_HOURS


def run_one(exp, label, cfg, seed, persist=True):
    key = run_key(cfg, seed)
    need_bank = cfg["save_bank"] and cfg["policy"] == "dqn"
    if persist and key in DONE and (not need_bank or os.path.exists(bank_path(key))):
        return DONE[key], True
    donors = None
    if cfg["donor"] is not None:
        dcfg = cfg_of(cfg["donor"])
        dkey = run_key(dcfg, seed)
        if not os.path.exists(bank_path(dkey)):
            run_one(exp, label + "/donor", dcfg, seed, persist)
        if os.path.exists(bank_path(dkey)):
            donors = torch.load(bank_path(dkey))
        else:
            rec = dict(key=key, exp=exp, label=label, seed=seed, cfg=cfg, skipped="donor society has no survivors")
            return rec, False
    res, bank = simulate(cfg, seed, donors)
    rec = dict(key=key, exp=exp, label=label, seed=seed, cfg=cfg, **res)
    if persist:
        os.makedirs(OUT_DIR, exist_ok=True)
        if bank is not None:
            torch.save(bank, bank_path(key))
        with open(os.path.join(OUT_DIR, RESULTS_FILE), "a") as fh:
            fh.write(json.dumps(rec) + "\n")
        DONE[key] = rec
    return rec, False


TABLE = ("pop", "cohort0_mortality", "deaths_starv_cum", "births_cum", "wealth_mean", "gini", "gini_ever_lived",
         "happy_mean", "act_collect", "act_invest", "act_reproduce")


def run_experiment(num, title, purpose, configs):
    print("=" * 100 + f"\nEXPERIMENT {num}: {title}\nPurpose: {purpose}\n" + "=" * 100, flush=True)
    rows, total, k = [], len(configs) * len(SEEDS), 0
    for label, cfg in configs:
        finals = []
        for seed in SEEDS:
            k += 1
            if over_budget():
                print(f"TIME_BUDGET_REACHED experiment={num} completed_runs={k - 1}/{total}", flush=True)
                return False
            rec, cached = run_one(f"E{num}", label, cfg, seed)
            if "skipped" in rec:
                print(f"[E{num}|{label}|seed={seed}] SKIPPED reason={rec['skipped']}", flush=True)
                continue
            s = rec["summaries"][-1]
            finals.append(s)
            print(f"[E{num}|{label}|seed={seed}] {'CACHED' if cached else 'DONE'} {k}/{total} pop={s['pop']} "
                  f"cohort0_mortality={s['cohort0_mortality']:.3f} wealth={s['wealth_mean']:.0f} gini={s['gini']:.3f} "
                  f"act_c/i/r={s['act_collect']:.2f}/{s['act_invest']:.2f}/{s['act_reproduce']:.2f} "
                  f"run_s={rec['wall']:.1f} elapsed_h={(time.time() - T_START) / 3600:.2f}", flush=True)
        rows.append((label, finals))
    print(f"SUMMARY EXPERIMENT {num} (final iteration, mean and sd over seeds)", flush=True)
    print("config | n | " + " | ".join(TABLE), flush=True)
    for label, finals in rows:
        if finals:
            cells = [f"{np.mean([s[m] for s in finals]):.3f} ({np.std([s[m] for s in finals], ddof=1) if len(finals) > 1 else 0.0:.3f})"
                     for m in TABLE]
            print(f"{label} | {len(finals)} | " + " | ".join(cells), flush=True)
    return True


def self_test():
    print("=" * 100 + "\nPRELIMINARY FUNCTIONAL TEST\nPurpose: Verify every code path on a reduced configuration.\n"
          + "=" * 100, flush=True)
    t0, n = time.time(), 0
    small = dict(n_agents=12, days=50, iterations=2, repro_wealth=60, cooldown=20, batch=8, buffer=64,
                 target_update=10, eps_decay_days=60)
    saved, LLM["obj"] = LLM["obj"], StubLLM()
    for env in ("paper", "ext"):
        for sname, sys_kw in SYSTEMS:
            for pol in ("dqn", "random", "rule", "forager", "llm"):
                c = cfg_of(sys_kw, EXT if env == "ext" else {}, small, policy=pol, save_bank=(pol == "dqn"),
                           inherit_net=(env == "ext"))
                res, bank = simulate(c, 7)
                s = res["summaries"][-1]
                ever = c["n_agents"] + s["births_cum"]
                assert s["pop"] == ever - s["deaths_cum"], "population accounting"
                assert all(math.isfinite(s[k]) for k in ("wealth_mean", "gini", "happy_mean")), "non-finite metric"
                assert abs(s["act_collect"] + s["act_invest"] + s["act_reproduce"] - 1.0) < 1e-9 or s["agent_steps"] == 0
                if pol == "dqn":
                    assert res["series"][-1]["loss"] is None or math.isfinite(res["series"][-1]["loss"])
                    if bank is not None:
                        r2, _ = simulate(cfg_of(c, policy="frozen", save_bank=False), 7, bank)
                        assert r2["summaries"][-1]["loss_mean"] is None
                        n += 1
                n += 1
    c = cfg_of(SYSTEMS[2][1], small)
    a, _ = simulate(c, 11)
    b, _ = simulate(c, 11)
    assert a["summaries"] == b["summaries"], "determinism"
    k = Bank(cfg_of(small))
    slots = [k.alloc() for _ in range(300)]
    assert len(set(slots)) == 300 and k.cap >= 300, "bank growth"
    LLM["obj"] = saved
    print(f"PRELIMINARY_TEST_RESULT cases={n + 2} status=ALL_PASSED wall_time_s={time.time() - t0:.1f}", flush=True)


def with_systems(extra, tag, systems=SYSTEMS):
    return [(f"{s}-{tag}", cfg_of(kw, extra)) for s, kw in systems]


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    print("=" * 100 + "\nENVIRONMENT\n" + "=" * 100, flush=True)
    print(f"python={platform.python_version()} torch={torch.__version__} numpy={np.__version__} "
          f"cuda_available={torch.cuda.is_available()} cpu_threads={torch.get_num_threads()} seeds={SEEDS} "
          f"output_dir={OUT_DIR} dqn_device=cpu", flush=True)
    self_test()
    load_done()
    P = with_systems
    ok = run_experiment(1, "FOUR SYSTEMS, MANUSCRIPT ENVIRONMENT, ALL POLICY TYPES",
                        "Reference runs in the batched engine; non-learning baselines under identical conditions.",
                        P(dict(save_bank=True), "dqn") + P(dict(policy="random"), "random")
                        + P(dict(policy="rule"), "rule") + P(dict(policy="forager"), "forager"))
    ok = ok and run_experiment(2, "MODEL-SPECIFICATION AND REWARD-DESIGN VARIANTS (MANUSCRIPT ENVIRONMENT)",
                               "Effect of state clipping, exploration clock, calendar, reproduction bonus, happiness term.",
                               P(dict(clip_state=True), "clipstate") + P(dict(eps_clock="global"), "epsglobal")
                               + P(dict(cooldown_clock="global"), "calglobal") + P(dict(r_repro=0.0), "noreprobonus")
                               + P(dict(happiness_term=False), "nohappy"))
    fc = SYSTEMS[:2]
    hp = [("lr", 1e-4), ("lr", 1e-2), ("gamma", 0.9), ("gamma", 0.999), ("eps_decay_days", 365),
          ("eps_decay_days", 1460), ("buffer", 200), ("buffer", 5000), ("batch", 16), ("batch", 64),
          ("hidden", 32), ("hidden", 128)]
    ok = ok and run_experiment(3, "ONE-AT-A-TIME HYPERPARAMETER SENSITIVITY (FREE AND CENTRALIZED)",
                               "Stability of outcomes under learning-rate, discount, exploration, buffer, batch, width changes.",
                               [x for k, v in hp for x in P({k: v}, f"{k}={v}", fc)])
    ok = ok and run_experiment(4, "INITIAL POPULATION SIZE AND RESOURCE SCARCITY (MANUSCRIPT ENVIRONMENT)",
                               "Persistence of the system ranking across initial population sizes and food supply.",
                               P(dict(n_agents=25), "n25") + P(dict(n_agents=100), "n100") + P(dict(n_agents=200), "n200")
                               + P(dict(food_lo=500, food_hi=1500), "food500-1500"))
    ext_dqn = dict(EXT, save_bank=True)
    grid = [(f"groups{g}-{sp}-ext-dqn", cfg_of(EXT, system="hybrid", groups=g, split=sp))
            for g in (1, 5, 10, 25) for sp in ("equal", "edu6040")]
    ok = ok and run_experiment(5, "CONSEQUENTIAL-DECISION ENVIRONMENT: POLICY TYPES, STRUCTURE AND ALLOCATION RULE",
                               "Daily subsistence for all agents, contribution-based pooling, symmetric investment, outcome-based reward.",
                               P(ext_dqn, "ext-dqn") + P(dict(EXT, inherit_net=True), "ext-dqn-inherit")
                               + P(dict(EXT, policy="random"), "ext-random") + P(dict(EXT, policy="rule"), "ext-rule")
                               + P(dict(EXT, policy="forager"), "ext-forager") + grid)
    three = [SYSTEMS[0], SYSTEMS[1], SYSTEMS[2]]
    tr = []
    for env_tag, env_kw in (("paper", {}), ("ext", EXT)):
        for sn, skw in three:
            donor = dict(skw, **env_kw, save_bank=True)
            for tn, tkw in three:
                tr.append((f"{env_tag}-train:{sn}-test:{tn}", cfg_of(tkw, env_kw, policy="frozen", donor=donor)))
    ok = ok and run_experiment(6, "PRETRAINED DQN AGENTS TRANSFERRED ACROSS SYSTEMS (FROZEN, GREEDY)",
                               "Agents trained in one system are placed, without further learning, in each system.", tr)
    if ok:
        name, err = None, None
        for cand in LLM_MODELS:
            try:
                LLM["obj"], name = LLMPolicy(cand), cand
                break
            except Exception as ex:
                err = repr(ex)[:300]
        if name is None:
            print("=" * 100 + f"\nEXPERIMENT 7: SKIPPED reason=language model unavailable error={err}\n" + "=" * 100,
                  flush=True)
        else:
            try:
                LLM["name"] = name
                t0 = time.time()
                LLM["obj"].probs([Society(cfg_of(policy="random")).prompt(Agent(0, BASE), 2000, [0, 1, 2])] * 128)
                rate = 128 / (time.time() - t0)
                est = 24 * 90000 / rate / 3600
                interval = 1 if est <= LLM_HOURS_MAX else int(math.ceil(est / LLM_HOURS_MAX))
                print(f"LLM_BENCHMARK model={name} device={LLM['obj'].dev} prompts_per_s={rate:.1f} "
                      f"estimated_hours_daily_decisions={est:.2f} decision_interval_days={interval}", flush=True)
                lk = dict(policy="llm", llm_interval=interval)
                run_experiment(7, "PRETRAINED LANGUAGE-MODEL AGENTS IN BOTH ENVIRONMENTS",
                               "Agents driven by a pretrained language model choose the daily action from a text description of their state.",
                               P(lk, "llm") + P(dict(EXT, **lk), "ext-llm"))
                print(f"LLM_COST prompts={LLM['obj'].n_prompts} seconds={LLM['obj'].seconds:.1f}", flush=True)
            except Exception as ex:
                print(f"EXPERIMENT 7 STOPPED error={repr(ex)[:300]}", flush=True)
    print("=" * 100 + f"\nRUN COMPLETE records={len(DONE)} elapsed_h={(time.time() - T_START) / 3600:.2f} "
          f"results_file={os.path.join(OUT_DIR, RESULTS_FILE)}\n" + "=" * 100, flush=True)


if __name__ == "__main__":
    main()


ENVIRONMENT
python=3.13.16 torch=2.11.0+cu130 numpy=2.1.3 cuda_available=True cpu_threads=2 seeds=[42, 123, 456] output_dir=/kaggle/working/dqn_society_run2 dqn_device=cpu
PRELIMINARY FUNCTIONAL TEST
Purpose: Verify every code path on a reduced configuration.
PRELIMINARY_TEST_RESULT cases=50 status=ALL_PASSED wall_time_s=5.6
RESUME records_found=0 file=/kaggle/working/dqn_society_run2/run2_results.jsonl
EXPERIMENT 1: FOUR SYSTEMS, MANUSCRIPT ENVIRONMENT, ALL POLICY TYPES
Purpose: Reference runs in the batched engine; non-learning baselines under identical conditions.
[E1|free-dqn|seed=42] DONE 1/48 pop=5 cohort0_mortality=0.920 wealth=3454 gini=0.561 act_c/i/r=0.25/0.51/0.24 run_s=5.0 elapsed_h=0.00
[E1|free-dqn|seed=123] DONE 2/48 pop=4 cohort0_mortality=0.960 wealth=2729 gini=0.393 act_c/i/r=0.22/0.55/0.23 run_s=4.2 elapsed_h=0.00
[E1|free-dqn|seed=456] DONE 3/48 pop=6 cohort0_mortality=0.920 wealth=3595 gini=0.393 act_c/i/r=0.23/0.51/0.25 run_s=4.5 elapsed_h=0.01
[E1|centralized-dqn

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM_BENCHMARK model=Qwen/Qwen2.5-1.5B-Instruct device=cuda prompts_per_s=20.5 estimated_hours_daily_decisions=29.25 decision_interval_days=6
EXPERIMENT 7: PRETRAINED LANGUAGE-MODEL AGENTS IN BOTH ENVIRONMENTS
Purpose: Agents driven by a pretrained language model choose the daily action from a text description of their state.
[E7|free-llm|seed=42] DONE 1/24 pop=0 cohort0_mortality=1.000 wealth=0 gini=0.000 act_c/i/r=0.00/0.00/0.00 run_s=25.4 elapsed_h=0.80
[E7|free-llm|seed=123] DONE 2/24 pop=0 cohort0_mortality=1.000 wealth=0 gini=0.000 act_c/i/r=1.00/0.00/0.00 run_s=25.3 elapsed_h=0.81
[E7|free-llm|seed=456] DONE 3/24 pop=0 cohort0_mortality=1.000 wealth=0 gini=0.000 act_c/i/r=0.00/0.00/0.00 run_s=18.2 elapsed_h=0.82
[E7|centralized-llm|seed=42] DONE 4/24 pop=51 cohort0_mortality=0.200 wealth=61160 gini=0.426 act_c/i/r=0.98/0.01/0.00 run_s=498.7 elapsed_h=0.95
[E7|centralized-llm|seed=123] DONE 5/24 pop=42 cohort0_mortality=0.280 wealth=54936 gini=0.423 act_c/i/r=0.99/0.01/0.00 run_s=